# Data Exploration — สำรวจข้อมูลก่อน preprocess

สำรวจ **dataset #1 (star schema)** ที่ยังไม่ผ่านการ preprocess เพื่อตอบว่า
**ทำไม** [`preprocessing.ipynb`](preprocessing.ipynb) ถึงตัดสินใจแบบที่ทำ

notebook นี้ไม่ได้มีไว้ดูกราฟสวย ๆ — ทุกหัวข้อจบด้วย **"ข้อสรุป → การตัดสินใจ"**
ที่ไปปรากฏจริงใน pipeline

| หัวข้อ | นำไปสู่การตัดสินใจอะไร |
|--------|----------------------|
| 1. ภาพรวมและความครบถ้วน | คอลัมน์ไหนใช้เป็น feature ได้บ้าง |
| 2. การแจกแจงคะแนน | ต้องใช้ `class_weight` ไหม |
| 3. คุณภาพข้อความ | ต้องล้าง HTML ไหม และตัดที่กี่คำ |
| 4. ความครอบคลุมตามเวลา | ตัดเส้น train/val/test ตรงไหน |
| 5. พฤติกรรมผู้รีวิว | `dim_user` ใช้ได้แค่ไหน |
| 6. การกระจายของสินค้า | ต้องตัดสินค้าที่รีวิวน้อยออกไหม |
| 7. ข้อความซ้ำ | มีสัญญาณรีวิวปลอมไหม |

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from elt.common import load_config, resolve_dataset  # noqa: E402

config = load_config()
SRC = resolve_dataset("elt", config)

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_review AS SELECT * FROM '{SRC}/fact_review/**/*.parquet'")
con.execute(f"CREATE VIEW review_text AS SELECT * FROM '{SRC}/review_text/**/*.parquet'")
con.execute(f"CREATE VIEW dim_product AS SELECT * FROM '{SRC}/dim_product.parquet'")
con.execute(f"CREATE VIEW dim_user   AS SELECT * FROM '{SRC}/dim_user.parquet'")
con.execute(f"CREATE VIEW dim_date   AS SELECT * FROM '{SRC}/dim_date.parquet'")

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 120)

print("โหลดข้อมูลแล้ว — ทุก view อ่านตรงจาก Parquet ไม่ได้โหลดเข้า RAM")

## 1. ภาพรวมและความครบถ้วนของข้อมูล

เริ่มจากคำถามพื้นฐานที่สุด: มีข้อมูลเท่าไหร่ และคอลัมน์ไหน**ใช้งานได้จริง**

In [ ]:
sizes = pd.DataFrame([
    (t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0])
    for t in ("fact_review", "review_text", "dim_product", "dim_user", "dim_date")
], columns=["table", "rows"])
sizes["rows"] = sizes["rows"].map("{:,}".format)
display(sizes)

con.sql("""
    SELECT category, count(*) AS n_reviews,
           count(DISTINCT product_key) AS n_products,
           count(DISTINCT user_key)    AS n_users,
           round(count(*)::DOUBLE / count(DISTINCT product_key), 1) AS reviews_per_product
    FROM fact_review GROUP BY 1 ORDER BY 2 DESC
""").df()

In [ ]:
# คอลัมน์ไหนมีค่าใช้ได้กี่ % — ตัวที่ต่ำมากจะใช้เป็น feature ไม่ได้
coverage = con.sql("""
    SELECT
        round(avg(has_metadata::INT), 4)             AS has_metadata,
        round(avg((product_title IS NOT NULL)::INT), 4) AS product_title,
        round(avg((store IS NOT NULL)::INT), 4)      AS store_brand,
        round(avg((price IS NOT NULL)::INT), 4)      AS price,
        round(avg((listed_avg_rating IS NOT NULL)::INT), 4) AS listed_avg_rating
    FROM dim_product
""").df().T.rename(columns={0: "สัดส่วนที่มีค่า (นับตามสินค้า)"})

# ถ่วงน้ำหนักตามรีวิว = สัดส่วนที่เหลือจริงเวลา join
weighted = con.sql("""
    SELECT round(avg((p.price IS NOT NULL)::INT), 4) AS price,
           round(avg((p.store IS NOT NULL)::INT), 4) AS store_brand
    FROM fact_review f JOIN dim_product p USING (product_key)
""").df().T.rename(columns={0: "สัดส่วนที่มีค่า (ถ่วงตามรีวิว)"})

display(coverage.join(weighted).fillna("-"))

print("→ ข้อสรุป: `price` มีค่าแค่ ~8% ของสินค้า / ~21% ของรีวิว")
print("  การตัดสินใจ: ใช้ price เป็น feature ได้เฉพาะกับ subset และต้องรายงานข้อจำกัดเสมอ")
print("  ส่วน store (แบรนด์) มีเกือบครบ ใช้ได้เต็มที่")

In [ ]:
# สำคัญ: ราคาที่ขาดหาย "สุ่ม" หรือไม่? ถ้าไม่สุ่ม การกรองทิ้งจะทำให้ผลเอน
con.sql("""
    SELECT CASE WHEN p.price IS NULL THEN 'ไม่รู้ราคา' ELSE 'รู้ราคา' END AS grp,
           count(*) AS n_reviews,
           round(avg(f.rating), 3)  AS avg_rating,
           round(avg(f.is_negative::INT), 3) AS negative_share,
           round(avg(f.verified_purchase::INT), 3) AS verified_share
    FROM fact_review f JOIN dim_product p USING (product_key)
    GROUP BY 1
""").df()

> **ข้อสรุปสำคัญ** — ถ้าคะแนนเฉลี่ยของสองกลุ่มต่างกันชัด แปลว่าราคาที่ขาดหาย
> **ไม่ได้ขาดหายแบบสุ่ม (MNAR)** การกรอง `price_band = 'Unknown'` ทิ้งจึงไม่ใช่แค่
> "ตัวอย่างเล็กลง" แต่ทำให้ผลลัพธ์**เอนไปทางบวก** — เขียนเตือนไว้ใน dataset card แล้ว

## 2. การแจกแจงคะแนน

**คำถาม** — คลาสสมดุลไหม ต้องจัดการอะไรก่อนเทรนโมเดล

In [ ]:
dist = con.sql("""
    SELECT rating, count(*) AS n_reviews,
           round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct
    FROM fact_review GROUP BY 1 ORDER BY 1
""").df()
display(dist)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(dist.rating, dist.n_reviews, color="steelblue")
ax1.set_title("การแจกแจงคะแนน"); ax1.set_xlabel("ดาว"); ax1.set_ylabel("จำนวนรีวิว")

by_cat = con.sql("""
    SELECT category, rating, count(*) * 100.0 /
           sum(count(*)) OVER (PARTITION BY category) AS pct
    FROM fact_review GROUP BY 1, 2
""").df()
by_cat.pivot(index="rating", columns="category", values="pct").plot(kind="bar", ax=ax2)
ax2.set_title("สัดส่วนคะแนนรายหมวด (%)"); ax2.set_xlabel("ดาว"); ax2.legend(fontsize=7)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

top = dist.loc[dist.n_reviews.idxmax()]
print(f"→ ข้อสรุป: คะแนน {int(top.rating)} ดาวคิดเป็น {top.pct:.1f}% ของทั้งหมด — เบ้ขวาหนัก")
print("  การตัดสินใจ: ตอนเทรน classifier ต้องใช้ class_weight='balanced'")
print("  และวัดผลด้วย macro-F1 ไม่ใช่ accuracy (ทายบวกหมดก็ได้ accuracy สูงแล้ว)")

## 3. คุณภาพของข้อความ — หัวใจของการตัดสินใจ preprocess

**คำถามสองข้อ**
1. ข้อความมี HTML ปนมากแค่ไหน → ต้องล้างไหม
2. รีวิวสั้นแค่ไหน → ควรตัดที่กี่คำ

In [ ]:
html_stats = con.sql("""
    SELECT
        count(*) AS total,
        sum((review_text LIKE '%<br%')::INT)   AS has_br_tag,
        sum((review_text LIKE '%&#3%')::INT)   AS has_numeric_entity,
        sum((review_text LIKE '%&amp;%')::INT) AS has_amp_entity,
        sum((review_text LIKE '%<%>%')::INT)   AS has_any_tag
    FROM review_text
""").df().T.rename(columns={0: "จำนวน"})
html_stats["สัดส่วน"] = (html_stats["จำนวน"] / html_stats.loc["total", "จำนวน"]).map("{:.2%}".format)
display(html_stats)

print("ตัวอย่างข้อความที่มี HTML ปน:\n")
for t, in con.execute("""
    SELECT review_text FROM review_text
    WHERE review_text LIKE '%<br%' OR review_text LIKE '%&#3%' LIMIT 3
""").fetchall():
    print("  ", t[:130].replace("\n", " "), "...\n")

> **ข้อสรุป → การตัดสินใจ** — ถ้าไม่ล้าง tokenizer จะนับ `br` เป็นคำจริง และ
> `&amp;` กลายเป็น token ขยะ กระทบทุกโมเดลที่ใช้ข้อความ
> จึงเขียน macro `clean_text()` ใน [`preprocessing.ipynb`](preprocessing.ipynb)
> โดย**ถอด `&amp;` เป็นลำดับสุดท้าย** เพื่อไม่ให้ `&amp;quot;` (escape ซ้อนสองชั้น)
> กลายเป็นเครื่องหมายคำพูดผิด ๆ

In [ ]:
# ความยาวก่อนล้าง (ประมาณการ — ความยาวจริงหลังล้างจะสั้นลงเล็กน้อย)
lens = con.sql("""
    SELECT
        length(review_text) - length(replace(review_text, ' ', '')) + 1 AS word_count
    FROM review_text USING SAMPLE 200000 ROWS (reservoir, 42)
""").df()

pcts = lens.word_count.quantile([.01, .05, .10, .25, .50, .75, .90, .99]).round(1)
display(pcts.rename("จำนวนคำ").to_frame().T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(lens.word_count.clip(upper=200), bins=60, color="steelblue")
ax1.set_title("การแจกแจงจำนวนคำ (ตัดที่ 200)"); ax1.set_xlabel("จำนวนคำ")

for thr in (1, 2, 3, 5, 10):
    ax2.bar(str(thr), (lens.word_count < thr).mean() * 100, color="indianred")
ax2.set_title("% รีวิวที่จะถูกตัดออก ถ้าตั้งเกณฑ์ขั้นต่ำ")
ax2.set_xlabel("เกณฑ์ขั้นต่ำ (คำ)"); ax2.set_ylabel("% ที่ถูกตัด")
plt.tight_layout(); plt.show()

for thr in (1, 2, 3, 5, 10):
    print(f"  ตัดที่ >= {thr:>2} คำ -> เสียข้อมูล {(lens.word_count < thr).mean():.2%}")

> **ข้อสรุป → การตัดสินใจ** — เลือก `min_words = 3` เพราะเป็นจุดที่ตัดรีวิวไร้สาระ
> ("ok", "good", "👍") ออกได้โดยเสียข้อมูลน้อยมาก ตั้งสูงกว่านี้จะเริ่มตัดรีวิวสั้นแต่มีสาระ
> อย่าง "ไซซ์เล็กไป" ซึ่งเป็น**ข้อมูลที่มีค่าที่สุด**สำหรับหมวดแฟชั่น

## 4. ความครอบคลุมตามเวลา — ตัดเส้น train/val/test ตรงไหน

**คำถาม** — ข้อมูลกระจายตามเวลาสม่ำเสมอไหม ถ้าไม่ ต้องปรับเส้นแบ่งอย่างไร

In [ ]:
yearly = con.sql("""
    SELECT d.year, count(*) AS n_reviews, round(avg(f.rating), 3) AS avg_rating,
           round(avg(f.is_negative::INT), 3) AS negative_share
    FROM fact_review f JOIN dim_date d USING (date_key)
    WHERE d.year >= 2010 GROUP BY 1 ORDER BY 1
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ["indianred" if y >= 2022 else "steelblue" for y in yearly.year]
ax1.bar(yearly.year, yearly.n_reviews, color=colors)
ax1.set_title("จำนวนรีวิวรายปี (แดง = เก็บไม่ครบ)"); ax1.set_ylabel("จำนวนรีวิว")

ax2.plot(yearly.year, yearly.avg_rating, marker="o", label="คะแนนเฉลี่ย")
ax2b = ax2.twinx()
ax2b.plot(yearly.year, yearly.negative_share, marker="s", color="indianred",
          label="สัดส่วนรีวิวลบ")
ax2.set_title("คะแนนเฉลี่ย vs สัดส่วนรีวิวเชิงลบ")
ax2.set_ylabel("คะแนนเฉลี่ย"); ax2b.set_ylabel("สัดส่วนรีวิวลบ")
ax2.legend(loc="upper left", fontsize=8); ax2b.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

display(yearly.tail(8))

peak = yearly.loc[yearly.n_reviews.idxmax()]
last = yearly.iloc[-1]
print(f"→ ปีที่มีรีวิวมากสุด: {int(peak.year)} ({peak.n_reviews:,.0f} รีวิว)")
print(f"  ปีสุดท้าย {int(last.year)}: {last.n_reviews:,.0f} รีวิว "
      f"({last.n_reviews / peak.n_reviews:.0%} ของปีสูงสุด)")

> **ข้อสรุป → การตัดสินใจ** — ปริมาณรีวิวร่วงหนักตั้งแต่ปี 2022 ซึ่งเป็น
> **ข้อจำกัดของการเก็บข้อมูลต้นทาง** (ชุดข้อมูลเก็บกลางปี 2023) ไม่ใช่ดีมานด์หายจริง
>
> ผลสองอย่าง:
> 1. ถ้าตัด val/test ที่ปี 2022/2023 จะได้ test แค่ ~1.7% → **เลื่อนเส้นมาหนึ่งปี**
>    เป็น val = 2021, test = 2022+ ได้สัดส่วน 77/14/10
> 2. **ห้ามอ่านกราฟซ้ายเป็น "ตลาดหดตัว"** — ถ้าจะดูแนวโน้มให้ใช้กราฟขวา
>    (คะแนนเฉลี่ย) ซึ่งไม่ขึ้นกับจำนวน
>
> สังเกตกราฟขวา: สัดส่วนรีวิวเชิงลบ**เพิ่มขึ้นจริง**ตามเวลา → นี่คือ distribution shift
> ที่โมเดลจะเจอตอน test ต้องเขียนเตือนไว้

## 5. พฤติกรรมผู้รีวิว — `dim_user` ใช้ได้แค่ไหน

**คำถาม** — feature ระดับผู้ใช้มีพลังอธิบายจริงไหม ถ้าคนส่วนใหญ่รีวิวครั้งเดียว
feature อย่าง `avg_rating_given` ก็แทบไม่มีข้อมูล

In [ ]:
seg = con.sql("""
    SELECT reviewer_segment,
           count(*) AS n_users,
           round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct_users,
           sum(lifetime_reviews) AS n_reviews,
           round(sum(lifetime_reviews) * 100.0 / sum(sum(lifetime_reviews)) OVER (), 2) AS pct_reviews,
           round(avg(avg_rating_given), 2) AS avg_rating_given
    FROM dim_user GROUP BY 1 ORDER BY n_reviews DESC
""").df()
display(seg)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.pie(seg.pct_users, labels=seg.reviewer_segment, autopct="%1.1f%%", startangle=90)
ax1.set_title("สัดส่วน 'จำนวนคน'")
ax2.pie(seg.pct_reviews, labels=seg.reviewer_segment, autopct="%1.1f%%", startangle=90)
ax2.set_title("สัดส่วน 'จำนวนรีวิว'")
plt.tight_layout(); plt.show()

oneoff = seg[seg.reviewer_segment == "One-off"].iloc[0]
print(f"→ ข้อสรุป: คนที่รีวิวครั้งเดียวจบคิดเป็น {oneoff.pct_users:.1f}% ของผู้ใช้ "
      f"และ {oneoff.pct_reviews:.1f}% ของรีวิวทั้งหมด")
print("  การตัดสินใจ: feature ระดับผู้ใช้ (user_lifetime_reviews, user_avg_rating_given)")
print("  มีพลังอธิบายจำกัด เพราะส่วนใหญ่ค่าจะเท่ากันหมด (1 รีวิว = ค่าเฉลี่ยคือคะแนนตัวเอง)")
print("  → ยังใส่เป็น feature ได้ แต่อย่าคาดหวังมาก และห้ามใช้ user_avg_rating_given")
print("    ทำนาย rating ของรีวิวเดียวกัน เพราะสำหรับ One-off มันคือคำตอบโดยตรง (leakage!)")

## 6. การกระจายของสินค้า — long tail แค่ไหน

**คำถาม** — สินค้าส่วนใหญ่มีรีวิวกี่อัน ต้องตัดสินค้าที่รีวิวน้อยออกก่อนวิเคราะห์ไหม

In [ ]:
prod_dist = con.sql("""
    SELECT
        CASE WHEN n <= 1 THEN '1'
             WHEN n <= 5 THEN '2-5'
             WHEN n <= 20 THEN '6-20'
             WHEN n <= 100 THEN '21-100'
             ELSE '100+' END AS review_bucket,
        count(*) AS n_products, sum(n) AS n_reviews
    FROM (SELECT product_key, count(*) AS n FROM fact_review GROUP BY 1)
    GROUP BY 1
""").df()
order = ["1", "2-5", "6-20", "21-100", "100+"]
prod_dist = prod_dist.set_index("review_bucket").reindex(order).reset_index()
prod_dist["pct_products"] = (prod_dist.n_products / prod_dist.n_products.sum() * 100).round(2)
prod_dist["pct_reviews"] = (prod_dist.n_reviews / prod_dist.n_reviews.sum() * 100).round(2)
display(prod_dist)

ax = prod_dist.set_index("review_bucket")[["pct_products", "pct_reviews"]].plot(kind="bar")
ax.set_title("สินค้า vs รีวิว แยกตามจำนวนรีวิวที่สินค้ามี")
ax.set_ylabel("%"); ax.set_xlabel("จำนวนรีวิวต่อสินค้า")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

few = prod_dist[prod_dist.review_bucket.isin(["1", "2-5"])].pct_products.sum()
print(f"→ ข้อสรุป: สินค้า {few:.1f}% มีรีวิวไม่เกิน 5 อัน — คะแนนเฉลี่ยของกลุ่มนี้เชื่อไม่ได้")
print("  การตัดสินใจ: ตอนจัดอันดับสินค้าหรือทำ clustering ต้องตั้งเกณฑ์ขั้นต่ำ")
print("  (ใช้ n_reviews >= 20 ใน model notebook) ไม่งั้นสินค้าที่มีรีวิวเดียว 5 ดาว")
print("  จะขึ้นอันดับ 1 ทุกครั้ง")

## 7. ข้อความซ้ำ — สัญญาณรีวิวปลอม

**คำถาม** — มีข้อความที่ซ้ำกันเป๊ะ ๆ หลายครั้งไหม ถ้ามีเยอะคือสัญญาณของรีวิวจ้าง
หรือ bot ซึ่งเป็นจุดตั้งต้นของคำถาม B5 (anomaly detection)

In [ ]:
dupes = con.sql("""
    WITH t AS (
        SELECT lower(trim(review_text)) AS txt, count(*) AS n
        FROM review_text
        WHERE length(review_text) >= 40      -- ข้อความสั้นซ้ำกันได้ตามธรรมชาติ
        GROUP BY 1 HAVING count(*) > 1
    )
    SELECT count(*) AS distinct_texts_repeated,
           sum(n) AS total_reviews_involved,
           max(n) AS max_repeat
    FROM t
""").df()
display(dupes)

total = con.execute("SELECT count(*) FROM review_text WHERE length(review_text) >= 40").fetchone()[0]
involved = int(dupes.total_reviews_involved.iloc[0] or 0)
print(f"รีวิวที่ข้อความซ้ำกับรีวิวอื่น: {involved:,} จาก {total:,} ({involved/total:.2%})\n")

print("ตัวอย่างข้อความที่ถูกใช้ซ้ำมากที่สุด:")
for txt, n in con.execute("""
    SELECT lower(trim(review_text)) AS txt, count(*) AS n
    FROM review_text WHERE length(review_text) >= 40
    GROUP BY 1 HAVING count(*) > 1 ORDER BY n DESC LIMIT 5
""").fetchall():
    print(f"  ใช้ซ้ำ {n:>4} ครั้ง: {txt[:95]}...")

> **ข้อสรุป → การตัดสินใจ** — ข้อความซ้ำเป๊ะจำนวนหนึ่งเป็นเรื่องปกติ (คำชมสั้น ๆ
> ที่คนเขียนเหมือนกันโดยบังเอิญ) แต่ข้อความยาวที่ซ้ำหลายสิบครั้งคือสัญญาณผิดปกติ
>
> **ไม่ได้ลบออกใน pipeline** เพราะเราไม่มีเฉลยว่าอันไหนปลอมจริง การลบพลาดจะทำให้
> ข้อมูลเสียหายกู้คืนไม่ได้ → เก็บไว้ทั้งหมด แล้วให้คำถาม **B5 (anomaly detection)**
> เป็นคนจัดการ ซึ่งผลลัพธ์คือ "น่าสงสัย" ไม่ใช่ "ปลอมแน่นอน"

---

## สรุป: EDA นำไปสู่การตัดสินใจอะไรบ้าง

| สิ่งที่พบ | การตัดสินใจใน pipeline |
|-----------|----------------------|
| ข้อความมี HTML ปนจำนวนมาก | เขียน macro `clean_text()` ถอด tag + entity |
| รีวิวสั้นมากมีอยู่จำนวนหนึ่ง | ตั้ง `min_words = 3` (เสียข้อมูลน้อย ตัดขยะได้) |
| คะแนนเบ้ขวาหนัก | ใช้ `class_weight='balanced'` + วัดด้วย macro-F1 |
| ปริมาณรีวิวปี 2022+ เก็บไม่ครบ | เลื่อนเส้น split เป็น val=2021 / test=2022+ |
| สัดส่วนรีวิวลบเพิ่มตามเวลา | เตือนเรื่อง distribution shift ใน dataset card |
| ราคาขาดหายแบบ MNAR | เตือนว่าการกรอง `Unknown` ทำให้ผลเอนบวก |
| ผู้รีวิวส่วนใหญ่รีวิวครั้งเดียว | feature ระดับผู้ใช้มีพลังจำกัด + ระวัง leakage |
| สินค้าส่วนใหญ่มีรีวิวน้อย | ตั้งเกณฑ์ `n_reviews >= 20` ก่อนจัดอันดับ/clustering |
| มีข้อความซ้ำผิดปกติ | ไม่ลบ แต่ยกให้ B5 anomaly detection จัดการ |

ขั้นต่อไป → [`preprocessing.ipynb`](preprocessing.ipynb) ที่ลงมือทำตามข้อสรุปเหล่านี้

In [ ]:
con.close()
print("exploration เสร็จเรียบร้อย")